# Agentic AI News Scraper: Professional Demonstration

This notebook presents a modular, agentic AI news scraper, designed for robust, extensible, and production-ready applications. It leverages advanced LLM-powered summarization, customizable data sources, and modern visualization and notification features.

**Contents:**
1. Python Module: Agent Classes
2. Import and Setup
3. Core Functionality Demonstration
4. Interactive Gradio User Interface

## 1. Python Module: Agent Classes

All core agent logic is encapsulated in `news_agent_module.py`, ensuring clean separation of concerns and reusability. This module defines the scanner, summarizer, and planner agents.

## 2. Import and Setup

Import the agent classes and initialize environment variables for secure API and email integration.

In [ ]:
# Import agent classes from the module
from news_agent_module import NewsScannerAgent, SummarizerAgent, NewsPlannerAgent

In [ ]:
# Optionally load environment variables for OpenAI/email
from dotenv import load_dotenv
load_dotenv()

## 3. Core Functionality Demonstration

This section demonstrates advanced usage, including semantic summarization (LLM), user-defined news sources, email digest (debug mode), and automated news visualization.

In [ ]:
# Advanced usage: Semantic summarization, customizable sources, email digest, and word cloud
import os

# Set use_openai=True to enable semantic summarization (requires OPENAI_API_KEY in .env)
summarizer = SummarizerAgent(use_openai=True)

# Customizable sources (user can modify this list)
sources = [
    "https://feeds.feedburner.com/ai-news",
    "https://www.aitrends.com/feed/",
    "https://www.artificialintelligence-news.com/feed/"
]
scanner = NewsScannerAgent(sources)
planner = NewsPlannerAgent(scanner, summarizer)

# Fetch and summarize articles
articles = planner.run()

# Display the first 3 articles with semantic summaries
articles[:3]

# Generate and display a word cloud of the news
planner.generate_wordcloud(articles, output_path="wordcloud.png")
from PIL import Image
import matplotlib.pyplot as plt
img = Image.open("wordcloud.png")
plt.imshow(img)
plt.axis('off')
plt.show()

# (Optional) Send email digest (requires EMAIL_ADDRESS and EMAIL_PASSWORD in .env)
# planner.send_email_digest(articles, to_email="your@email.com")

In [ ]:
# Example usage: Fetch and display latest AI news
sources = [
    "https://feeds.feedburner.com/ai-news",
    "https://www.aitrends.com/feed/",
    "https://www.artificialintelligence-news.com/feed/"
]
scanner = NewsScannerAgent(sources)
summarizer = SummarizerAgent()
planner = NewsPlannerAgent(scanner, summarizer)

# Fetch and summarize articles
demo_articles = planner.run()

# Display the first 3 articles
demo_articles[:3]

## 4. Interactive Gradio User Interface

The following Gradio UI enables:
- Custom input of news sources
- Toggle for LLM-based semantic summarization
- Automated word cloud visualization
- (Debug) Email digest preview

This interface is suitable for both technical and non-technical stakeholders.

### Gradio UI: Professional Features

- Input custom RSS sources for flexible data acquisition
- Enable/disable advanced LLM summarization
- Visualize news trends with a word cloud
- (Debug) Preview email digest content for notification workflows

In [ ]:
import gradio as gr
from news_agent_module import NewsScannerAgent, SummarizerAgent, NewsPlannerAgent
import os

def gradio_news_agent(sources, use_openai, email):
    sources_list = [s.strip() for s in sources.split(',') if s.strip()]
    summarizer = SummarizerAgent(use_openai=use_openai)
    scanner = NewsScannerAgent(sources_list)
    planner = NewsPlannerAgent(scanner, summarizer)
    articles = planner.run()
    # News HTML
    if not articles:
        news_html = "No news found. Try again later."
    else:
        news_html = ""
        for art in articles[:10]:
            news_html += f"<b>{art['title']}</b><br>"
            news_html += f"<i>{art['pub_date']}</i><br>"
            news_html += f"<a href='{art['link']}' target='_blank'>Read more</a><br>"
            news_html += f"{art['summary']}<br><hr>"
    # Email
    email_status = ""
    if email:
        try:
            planner.send_email_digest(articles, to_email=email)
            email_status = f"Digest sent to {email}"
        except Exception as e:
            email_status = f"Email failed: {e}"
    return news_html, None, email_status

default_sources = "https://feeds.feedburner.com/ai-news, https://www.aitrends.com/feed/, https://www.artificialintelligence-news.com/feed/"
demo = gr.Interface(
    fn=gradio_news_agent,
    inputs=[
        gr.Textbox(label="News RSS Sources (comma-separated)", value=default_sources),
        gr.Checkbox(label="Use OpenAI Semantic Summarization", value=False),
        gr.Textbox(label="Email for Digest (optional)", value="")
    ],
    outputs=[
        gr.HTML(label="Latest AI News"),
        gr.Image(type="pil", label="Word Cloud"),
        gr.Textbox(label="Email Status")
    ],
    title="Agentic AI News Scraper (Advanced)",
    description="Customize sources, enable semantic summarization, visualize news, and get a digest by email."
)
demo.launch()